# message的使用
## 1、消息格式
举例：json格式

In [ ]:
import os

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL
)


In [ ]:

messages=[
    {"role":"system","content":"你是一个友好的ai助手"},
    {"role":"user","content":"1+2=？"},
    {"role":"assistant","content":"3"},
    {"role":"user","content":"我刚才问了什么问题"}
]
response=model.invoke(messages)
print(response)

举例2：消息对象列表

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
messages=[
    SystemMessage("你是一个友好的ai助手"),
    HumanMessage("1+2=?"),
    AIMessage("3"),
    HumanMessage("我刚才问了什么问题"),
]
response=model.invoke(messages)
print(response)

# 2、humanmessage的使用
举例1：openrouter平台为例

In [ ]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
import os

load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = ChatOpenRouter(
    # model="openai/gpt-5.4-mini",
    model="openai/gpt-4o-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

messages = [
    SystemMessage(
        """你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息
可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出
JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出
格式：{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}"""
    ),
    HumanMessage(
        content="我认为 1+1=2",
        name="Bob"
    ),
    HumanMessage(
        content="我认为 1+1>2",
        name="Tom"
    ),
    HumanMessage(
        content="请列出谁说了什么，不要判断对错。",
        name="audience"
    )
]

response = model.invoke(messages)
print(response.content)

## 3、AIMessage的使用
举例：

In [ ]:
import os
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL
)
messages = [
SystemMessage("你叫小智，是一名助人为乐的助手。"),
HumanMessage("你好，好久不见，请介绍下你自己。")
]
response = model.invoke(messages)
rprint(response)

In [ ]:
print(response.usage_metadata)

## 5、对话历史优化
举例：

In [ ]:
def keep_recent_messages(messages,max_pairs=3):
    """保留最近的n轮对话
    max_pairs:保留对话的轮数(每轮=user+assitanta)
    """
    #分离system消息和对话消息
    system_messages = [m for m in messages if m.get("role")=="system"]
    conversation_messages = [m for m in messages if m.get("role")!="system"]

    recent_messages = conversation_messages[-(2*max_pairs):]
    return system_messages+recent_messages


In [ ]:
# 初始化对话历史
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]

# 第 1 轮对话
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})

# 第 2 轮对话
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

# 第 3 轮对话
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})

print(f"原始消息数: {len(long_conversation)}")

# 优化历史消息：只保留最近 2 轮对话
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")

# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})

# 使用裁剪后的对话历史发起请求
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

## 6、多轮对话聊天机器人

In [ ]:
import os

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
EXIT_WORD="quit"
MAX_PAIRS_HISTORY=10
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL
)
#3.维护一个消息列表
messages=[
    {
        "role":"system","content":"你是一个耐心，友好的ai助手，你叫小八姐姐，可以回答学生提出的问题"
    }
]
print(f"请输入具体的问题，当输入{EXIT_WORD}时就结束对话")
i=1
while True:
    print("\n","="*10,f"第{i}轮对话开始","="*10,"\n")
    user_input=input("请输入:")
    #判断是否结束当前会话
    if user_input=="quit":
        print("会话结束，下次再来")
        break
    #将用户信息添加到消息列表中
    messages.append(
        {"role":"user","content":user_input}
    )
    print("小八姐姐",end="",flush=True)
    #拼接ai回复的消息
    reply_content=""
    #优化历史记忆
    memory_messages=keep_recent_messages(messages,MAX_PAIRS_HISTORY)
    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content,end="",flush=True)
            reply_content+=chunk.content
    print("\n","="*10,f"第{i}轮对话结束","="*10,"\n")
    i+=1
    #将模型的响应添加到消息列表
    messages.append({"role":"assistant","content":reply_content})